In [ ]:
# IMPORTAR BIBLIOTECAS ---
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# ROOT DO PROJETO
PROJECT_ROOT = Path(__file__).resolve().parents[3]

# PASTA DOS CSVs ANALÍTICOS
OUTPUT_DIR = (PROJECT_ROOT/ "scripts"/ "Analytics"/ "outputs"/ "gold_01")

# PASTA DOS GRÁFICOS
GRAFICOS_DIR = (PROJECT_ROOT/ "scripts"/ "Analytics"/ "graficos"/ "gold_01")
GRAFICOS_DIR.mkdir(parents=True,exist_ok=True)
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# GRÁFICO 1 - RINCIPAIS CARGOS
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# CARREGA ESTRUTURA ATUAL DOS CARGOS ---
df_cargos = pd.read_csv(OUTPUT_DIR / "cargos_estrutura_atual.csv")
df_cargos

# TIRAR "OUTRA OPÇAO" ---
df_cargos_grafico = (df_cargos
    .loc[df_cargos["cargo_harmonizado"] != "Outra Opção"]
    .copy()
)
df_cargos_grafico

# TOP 5 CARGOS ---
df_cargos_top5 = (df_cargos_grafico
    .sort_values("pct_na_dimensao",ascending=False)
    .head(5)
    .copy()
)
df_cargos_top5

# NOMES CURTOS PARA APRESENTAÇÃO ---
nomes_cargos = {
    "Analista de Dados/Data Analyst" : "Analista de Dados",
    "Cientista de Dados/Data Scientist" : "Cientista de Dados",
    "Analista de BI/BI Analyst" : "Analista de BI",
    "Analista de Negócios/Business Analyst" : "Analista de Negócios",
    "Engenharia e Arquitetura de Dados" : "Engenharia e Arquitetura",
    "Analytics Engineer" : "Analytics Engineer"
}
df_cargos_top5["cargo_exibicao"] = (df_cargos_top5["cargo_harmonizado"].replace(nomes_cargos))
df_cargos_top5

# ORDENA PARA O MAIOR APARECER NO TOPO ---
df_cargos_top5 = (df_cargos_top5
    .sort_values("pct_na_dimensao",ascending=True)
)
df_cargos_top5

# CONCENTRAÇÃO DOS 3 PRINCIPAIS CARGOS ---
participacao_top3 = (df_cargos
    .sort_values("pct_na_dimensao",ascending=False)
    .head(3)["pct_na_dimensao"]
    .sum()
)
# -------------------------

# CRIA GRÁFICO ---
# -------------------------
fig, ax = plt.subplots(figsize=(11, 6))

ax.barh(
    df_cargos_top5["cargo_exibicao"],
    df_cargos_top5["pct_na_dimensao"]
)

# TÍTULO ---
ax.set_title(
    "Principais cargos entre os respondentes | Edição 2025-2026",
    fontsize=17,
    fontweight="bold",
    loc="left",
    pad=28
)

# SUBTÍTULO ---
ax.text(
    0,
    1.03,
    "Análise de Dados lidera a estrutura ocupacional da amostra",
    transform=ax.transAxes,
    fontsize=11
)

ax.text(
    0.98,
    1.03,
    f"TOP 3: {participacao_top3:.1f}%",
    transform=ax.transAxes,
    fontsize=11,
    fontweight="bold",
    ha="right"
)


# RÓTULOS DOS PERCENTUAIS ---
for i, valor in enumerate(
    df_cargos_top5["pct_na_dimensao"]
):
    ax.text(
        valor + 0.35,
        i,
        f"{valor:.1f}%",
        va="center",
        fontsize=11,
        fontweight="bold"
    )

# LIMPEZA VISUAL ---
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_xticks([])

ax.tick_params(
    axis="y",
    length=0,
    labelsize=11
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["bottom"].set_visible(False)
ax.spines["left"].set_visible(False)


# ESPAÇO À DIREITA DOS RÓTULOS ---
ax.set_xlim(
    0,
    df_cargos_top5["pct_na_dimensao"].max() + 4
)

# NOTA SOBRE OUTROS ---
fig.text(
    0.01,
    0.01,
    "Nota: 'Outra Opção' (8,2%) não é exibida por não representar um cargo específico.",
    fontsize=8
)
plt.tight_layout(rect=[0, 0.05, 1, 1])
# -------------------------


# SALVA GRÁFICO
# -------------------------
plt.savefig(
    GRAFICOS_DIR / "01_estrutura_atual_cargos.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------



# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# GRÁFICO 2 - EVOLUÇÃO DOS PRINCIPAIS CARGOS
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# CARREGA HISTÓRICO DOS CARGOS
df_cargos_historico = pd.read_csv(
    OUTPUT_DIR / "cargos_historico.csv"
)
print(df_cargos_historico)


cargos_destaque = [
    "Analista de Dados/Data Analyst",
    "Engenharia e Arquitetura de Dados",
    "Cientista de Dados/Data Scientist",
    "Analista de BI/BI Analyst"
]

df_evolucao_cargos = (df_cargos_historico
    .loc[df_cargos_historico["cargo_harmonizado"].isin(cargos_destaque)]
    .copy()
)

# NOMES CURTOS
nomes_cargos = {
    "Analista de Dados/Data Analyst" : "Analista de Dados",
    "Engenharia e Arquitetura de Dados" : "Engenharia e Arquitetura",
    "Cientista de Dados/Data Scientist" : "Cientista de Dados",
    "Analista de BI/BI Analyst" : "Analista de BI"
}

df_evolucao_cargos["cargo_exibicao"] = (df_evolucao_cargos["cargo_harmonizado"].replace(nomes_cargos))

periodos = ["2023-2024","2024-2025","2025-2026"]


# AJUSTES MANUAIS PARA EVITAR SOBREPOSIÇÃO
offset_inicio = {
    "Analista de Dados": 0,
    "Engenharia e Arquitetura": 0.35,
    "Cientista de Dados": -0.35,
    "Analista de BI": 0
}

offset_final = {
    "Analista de Dados": 0,
    "Engenharia e Arquitetura": 0.35,
    "Cientista de Dados": -0.35,
    "Analista de BI": 0
}
# -------------------------


# CRIA GRÁFICO
# -------------------------
fig, ax = plt.subplots(figsize=(12, 6))

for _, linha in df_evolucao_cargos.iterrows():

    cargo = linha["cargo_exibicao"]

    valores = [
        linha["2023-2024"],
        linha["2024-2025"],
        linha["2025-2026"]
    ]

    variacao = (
        linha["2025-2026"]
        - linha["2023-2024"]
    )

    ax.plot(
        periodos,
        valores,
        marker="o",
        linewidth=2.5
    )


    # VALOR INICIAL
    ax.text(
        -0.05,
        valores[0] + offset_inicio[cargo],
        f"{valores[0]:.1f}%",
        ha="right",
        va="center",
        fontsize=9
    )


    # NOME + VALOR FINAL + VARIAÇÃO
    ax.text(
        2.04,
        valores[-1] + offset_final[cargo],
        (
            f"{cargo}  "
            f"{valores[-1]:.1f}%  "
            f"({variacao:+.1f} p.p.)"
        ),
        ha="left",
        va="center",
        fontsize=10,
        fontweight="bold"
    )

# TÍTULO
ax.set_title(
    "Evolução dos principais cargos | 2023-2024 a 2025-2026",
    fontsize=17,
    fontweight="bold",
    loc="left",
    pad=28
)

# SUBTITULO
ax.text(
    0,
    1.03,
    "BI perde 4,5 p.p.; principais carreiras permanecem estáveis",
    transform=ax.transAxes,
    fontsize=11
)

# LIMPEZA VISUAL
ax.set_xlabel("")
ax.set_ylabel("")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_visible(False)

ax.tick_params(
    axis="y",
    left=False,
    labelleft=False
)

ax.tick_params(
    axis="x",
    length=0,
    labelsize=10
)

# MAIS ESPAÇO PARA OS RÓTULOS À DIREITA
ax.set_xlim(
    -0.15,
    3.1
)
plt.tight_layout()
# -------------------------


# SALVA GRÁFICO
# -------------------------
plt.savefig(
    GRAFICOS_DIR / "02_evolucao_principais_cargos.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------



# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# GRÁFICO 3 - ESTRUTURA ATUAL POR NÍVEL
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# CARREGA TABELA DE NÍVEL ATUAL
df_nivel = pd.read_csv( OUTPUT_DIR / "nivel_atual.csv")


# ORDENA PARA O MAIOR FICAR NO TOPO
df_nivel_grafico = (df_nivel
    .sort_values("pct_na_dimensao",ascending=True)
    .copy()
)
# -------------------------


# CRIA GRÁFICO
# -------------------------
fig, ax = plt.subplots(figsize=(10, 5.5))

ax.barh(
    df_nivel_grafico["valor"],
    df_nivel_grafico["pct_na_dimensao"]
)

# TÍTULO
ax.set_title(
    "Distribuição por nível profissional | Edição 2025-2026",    
    fontsize=17,
    fontweight="bold",
    loc="left",
    pad=28
)

# SUBUTITULO
ax.text(
    0,
    1.03,
    "Quase metade da amostra está em níveis Sênior ou Staff+",
    transform=ax.transAxes,
    fontsize=11
)

# CALLOUT
pct_senior_staff = (df_nivel
    .loc[df_nivel["valor"].isin(["Sênior", "Especialista/Staff+"]),"pct_na_dimensao"]
    .sum()
)

ax.text(
    0.98,
    1.03,
    f"SÊNIOR + STAFF+: {pct_senior_staff:.1f}%",
    transform=ax.transAxes,
    fontsize=11,
    fontweight="bold",
    ha="right"
)

# RÓTULOS
for i, valor in enumerate(
    df_nivel_grafico["pct_na_dimensao"]
):
    ax.text(
        valor + 0.4,
        i,
        f"{valor:.1f}%",
        va="center",
        fontsize=11,
        fontweight="bold"
    )

# LIMPEZA VISUAL
ax.set_xlabel("")
ax.set_ylabel("")

ax.set_xticks([])

ax.tick_params(
    axis="y",
    length=0,
    labelsize=11
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["bottom"].set_visible(False)
ax.spines["left"].set_visible(False)


ax.set_xlim(
    0,
    df_nivel_grafico["pct_na_dimensao"].max() + 5
)

# NOTA - ESPECIALISTAS/STAFF+
fig.text(
    0.01,
    0.01,
    "Nota: a categoria Especialista/Staff+ passou a existir na edição 2025-2026.",
    fontsize=8
)
plt.tight_layout(rect=[0, 0.05, 1, 1])
# -------------------------


# SALVA GRÁFICO
# -------------------------
plt.savefig(
    GRAFICOS_DIR / "03_estrutura_atual_nivel.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------



# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# GRÁFICO 4 - PRINCIPAIS SETORES
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# CARREGA TABELA DE SETORES ATUAL
df_setor = pd.read_csv(OUTPUT_DIR / "setor_atual.csv")

# REMOVE "OUTRA OPÇÃO" E SELECIONA TOP 6
df_setor_grafico = (df_setor
    .loc[df_setor["valor"] != "Outra Opção"]
    .sort_values("pct_na_dimensao",ascending=False)
    .head(6)
    .copy()
)

# NOMES MAIS CURTOS PARA APRESENTAÇÃO
nomes_setores = {
    "Finanças ou Bancos" : "Finanças e Bancos",
    "Tecnologia/Fábrica de Software" : "Tecnologia e Software",
    "Área de Consultoria" : "Consultoria"
}
df_setor_grafico["setor_exibicao"] = (df_setor_grafico["valor"].replace(nomes_setores))

# ORDENA PARA O MAIOR FICAR NO TOPO
df_setor_grafico = (df_setor_grafico
    .sort_values("pct_na_dimensao",ascending=True)
)

# CONCENTRAÇÃO DOS DOIS PRINCIPAIS SETORES
participacao_top2 = (df_setor
    .loc[df_setor["valor"] != "Outra Opção"]
    .sort_values("pct_na_dimensao",ascending=False)
    .head(2)["pct_na_dimensao"]
    .sum()
)
# -------------------------


# CRIA GRÁFICO
# -------------------------
fig, ax = plt.subplots(figsize=(10, 5.5))

ax.barh(
    df_setor_grafico["setor_exibicao"],
    df_setor_grafico["pct_na_dimensao"]
)


# TÍTULO
ax.set_title(
    "Principais setores de atuação | Edição 2025-2026",
    fontsize=17,
    fontweight="bold",
    loc="left",
    pad=28
)

# SUBTITULO
ax.text(
    0,
    1.03,
    "Finanças e Tecnologia concentram mais de um terço da amostra",    
    transform=ax.transAxes,
    fontsize=11
)

# CALLOUT
ax.text(
    0.98,
    1.03,
    f"TOP 2: {participacao_top2:.1f}%",
    transform=ax.transAxes,
    fontsize=11,
    fontweight="bold",
    ha="right"
)

# RÓTULOS
for i, valor in enumerate(
    df_setor_grafico["pct_na_dimensao"]
):
    ax.text(
        valor + 0.3,
        i,
        f"{valor:.1f}%",
        va="center",
        fontsize=11,
        fontweight="bold"
    )

# LIMPEZA VISUAL
ax.set_xlabel("")
ax.set_ylabel("")

ax.set_xticks([])

ax.tick_params(
    axis="y",
    length=0,
    labelsize=11
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["bottom"].set_visible(False)
ax.spines["left"].set_visible(False)

ax.set_xlim(0,df_setor_grafico["pct_na_dimensao"].max() + 4)

# NOTA - OUTRA OPC
fig.text(
    0.01,
    0.01,
    "Nota: 'Outra Opção' (7,5%) não é exibida por não representar um setor específico.",
    fontsize=8
)
plt.tight_layout(rect=[0, 0.05, 1, 1])
# -------------------------


# SALVA GRÁFICO
# -------------------------
plt.savefig(
    GRAFICOS_DIR / "04_principais_setores.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------



# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# GRÁFICO 5 - EVOLUÇÃO DO MODELO DE TRABALHO
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# CARREGA TABELA CONSOLIDADA
df_modelo = pd.read_csv(OUTPUT_DIR / "modelo_trabalho_consolidado.csv")

# ORGANIZA OS DADOS
df_modelo_grafico = (df_modelo
    .pivot(index="edicao",columns="grupo_modelo",values="pct_na_dimensao")
    .reindex(["2024-2025", "2025-2026"])
)

# VARIAÇÕES
variacao_remoto = (df_modelo_grafico.loc["2025-2026", "100% remoto"] - df_modelo_grafico.loc["2024-2025", "100% remoto"])

variacao_presencial = (df_modelo_grafico.loc["2025-2026","Com componente presencial"]- df_modelo_grafico.loc["2024-2025","Com componente presencial"])
# -------------------------


# CRIA GRÁFICO
# -------------------------
fig, ax = plt.subplots(figsize=(11, 5.5))

# PRIMEIRA PARTE DA BARRA
ax.barh(
    df_modelo_grafico.index,
    df_modelo_grafico["100% remoto"],
    label="100% remoto"
)

# SEGUNDA PARTE DA BARRA
ax.barh(
    df_modelo_grafico.index,
    df_modelo_grafico["Com componente presencial"],
    left=df_modelo_grafico["100% remoto"],
    label="Com componente presencial"
)

# RÓTULOS DENTRO DAS BARRAS
for i, edicao in enumerate(
    df_modelo_grafico.index
):

    remoto = (
        df_modelo_grafico
        .loc[edicao, "100% remoto"]
    )

    presencial = (
        df_modelo_grafico
        .loc[
            edicao,
            "Com componente presencial"
        ]
    )


    # REMOTO
    ax.text(
        remoto / 2,
        i,
        f"100% remoto\n{remoto:.1f}%",
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold"
    )


    # COMPONENTE PRESENCIAL
    ax.text(
        remoto + presencial / 2,
        i,
        (
            "Com componente presencial\n"
            f"{presencial:.1f}%"
        ),
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold"
    )

# TÍTULO
ax.set_title(
    "Distribuição dos modelos de trabalho entre os respondentes",
    fontsize=17,
    fontweight="bold",
    loc="left",
    pad=28
)

# SUBTITULO
ax.text(
    0,
    1.03,    
    "Modelos com presença ganham 6,0 p.p.; remoto perde participação",
    transform=ax.transAxes,
    fontsize=11
)

# CALLOUT
ax.text(
    0.98,
    1.03,
    (
        f"REMOTO: {variacao_remoto:+.1f} p.p.  |  "
        f"PRESENCIAL: {variacao_presencial:+.1f} p.p."
    ),
    transform=ax.transAxes,
    fontsize=10,
    fontweight="bold",
    ha="right"
)

# LIMPEZA VISUAL
ax.set_xlim(0,100)

ax.set_xlabel("")
ax.set_ylabel("")

ax.set_xticks([])

ax.tick_params(
    axis="y",
    length=0,
    labelsize=11
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["bottom"].set_visible(False)
ax.spines["left"].set_visible(False)

plt.tight_layout(rect=[0, 0.05, 1, 1])

# NOTA SOBRE VARIAVEL
fig.text(
    0.01,
    0.01,
    "Nota: a variável está disponível apenas nas edições 2024-2025 e 2025-2026.",
    fontsize=8
)
# -------------------------


# SALVA GRÁFICO
# -------------------------
plt.savefig(
    GRAFICOS_DIR / "05_evolucao_modelo_trabalho.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------



# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# GRÁFICO 6 - SITUAÇÃO DE TRABALHO
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# CARREGA TABELA HISTÓRICA
df_situacao = pd.read_csv(OUTPUT_DIR / "situacao_trabalho_historico.csv")

categorias_destaque = [
    "Empregado (CLT)",
    "Empreendedor ou Empregado (CNPJ)",
    "Desempregado, buscando recolocação",
    "Vivo no Brasil e trabalho remoto para empresa de fora do Brasil",
    "Servidor Público"
]

df_situacao_grafico = (df_situacao
    .loc[df_situacao["valor"].isin(categorias_destaque)]
    .copy()
)

# NOMES CURTOS
nomes_situacao = {
    "Empregado (CLT)" : "Empregado (CLT)",
    "Empreendedor ou Empregado (CNPJ)" : "CNPJ",
    "Desempregado, buscando recolocação" : "Buscando recolocação",
    "Vivo no Brasil e trabalho remoto para empresa de fora do Brasil" : "Remoto para empresa estrangeira",
    "Servidor Público" : "Servidor Público"
}
df_situacao_grafico["situacao_exibicao"] = (df_situacao_grafico["valor"].replace(nomes_situacao))

# ORDENA PELA VARIAÇÃO
df_situacao_grafico = (df_situacao_grafico
    .sort_values("variacao_pp",ascending=True)
    .reset_index(drop=True)
)

# SEPARA GANHOS E QUEDAS
df_quedas = (df_situacao_grafico
    .loc[df_situacao_grafico["variacao_pp"] < 0]
)

df_crescimentos = (df_situacao_grafico
    .loc[df_situacao_grafico["variacao_pp"] >= 0]
)
# -------------------------


# CRIA GRÁFICO
# -------------------------
fig, ax = plt.subplots(figsize=(10.5, 5.5))

# QUEDAS
ax.barh(
    df_quedas["situacao_exibicao"],
    df_quedas["variacao_pp"]
)

# CRESCIMENTOS
ax.barh(
    df_crescimentos["situacao_exibicao"],
    df_crescimentos["variacao_pp"]
)

# LINHA DE REFERÊNCIA
ax.axvline(0,linewidth=1)

# RÓTULOS
for _, linha in df_situacao_grafico.iterrows():

    variacao = linha["variacao_pp"]
    atual = linha["2025-2026"]
    categoria = linha["situacao_exibicao"]

    if variacao >= 0:

        ax.text(
            variacao + 0.12,
            categoria,
            f"+{variacao:.1f} p.p.  |  {atual:.1f}% atual",
            va="center",
            ha="left",
            fontsize=10,
            fontweight="bold"
        )

    else:

        ax.text(
            variacao - 0.12,
            categoria,
            f"{variacao:.1f} p.p.",
            va="center",
            ha="right",
            fontsize=10,
            fontweight="bold"
        )

        ax.text(
            0.15,
            categoria,
            f"{atual:.1f}% atual",
            va="center",
            ha="left",
            fontsize=9
        )

# TITULO
ax.set_title(
    "Situaçaão de Trabalo| 2024-2025 à 2025-2026",
    fontsize=17,
    fontweight="bold",
    loc="left",
    pad=28
)

# SUBTITULO
ax.text(
    0,
    1.03,
    "CLT perde espaço; CNPJ e trabalho internacional ganham participação",
    transform=ax.transAxes,
    fontsize=11
)

# LIMPEZA VISUAL
ax.set_xlabel("")
ax.set_ylabel("")

ax.set_xticks([])

ax.tick_params(
    axis="y",
    length=0,
    labelsize=10
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["bottom"].set_visible(False)
ax.spines["left"].set_visible(False)

ax.set_xlim(
    df_situacao_grafico["variacao_pp"].min() - 2,
    df_situacao_grafico["variacao_pp"].max() + 4
)
plt.tight_layout()
# -------------------------


# SALVA GRÁFICO
# -------------------------
plt.savefig(
    GRAFICOS_DIR / "06_variacao_situacao_trabalho.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()